# HDD 25 — Q4 blind protocol and candidate lock

Notebook 24 passed an offline/streaming parity replay on 50 Q3 drives. Freeze the research candidate and comparison rules **before reading Q4**. Q3 development outcomes were inspected repeatedly; these choices are therefore informed by Q3 and must be judged on a genuinely separate Q4 future-test drive cohort. This notebook reads only existing Q3 development artifacts, checkpoint hashes and the July 1 drive reservation. It never opens the Q4 archive or failure flags.

The primary self-supervised candidate is the frozen **sector reconstruction error** for every model, using Notebook 18’s model-specific 99.8th-percentile calibration threshold. It is compared to the previously frozen five-model Isolation Forest and simple SMART sector rules on identical scoreable Q4 endpoints. Temperature and joint reconstruction are secondary channels, reported without replacing the primary after seeing Q4. Regime and drift scores are excluded because Notebooks 21–22 did not validate them. No winner is declared here, and nothing is deployed automatically.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
AUDIT_ID='q3_multimodel_event_coverage_v1'
PILOT_ID='q3_five_model_isolation_forest_v1'
COMPARE_ID='q3_matched_detector_baseline_comparison_v1'
TRAIN_ID='q3_five_model_masked_transformer_v1'
SCORE_ID='q3_five_model_transformer_scoring_v1'
REGIME_AUDIT_ID='q3_regime_stability_smart_audit_v1'
ABLATION_ID='q3_nuisance_controlled_regime_ablation_v1'
SENS_ID='q3_frozen_detector_synthetic_sensitivity_v1'
REPLAY_ID='q3_frozen_streaming_inference_replay_v1'
LOCK_ID='q4_blind_candidate_protocol_v1'
ROOT=COMPONENT_ROOT/'results'/RUN_ID
AUDIT=ROOT/AUDIT_ID;PILOT=ROOT/PILOT_ID;COMPARE=ROOT/COMPARE_ID
TRAIN=ROOT/TRAIN_ID;SCORE=ROOT/SCORE_ID
REGIME_AUDIT=ROOT/REGIME_AUDIT_ID;ABLATION=ROOT/ABLATION_ID
SENS=ROOT/SENS_ID;REPLAY=ROOT/REPLAY_ID;OUT=ROOT/LOCK_ID
OUT.mkdir(parents=True,exist_ok=True)
TRAINED=COMPONENT_ROOT/'models/trained'/RUN_ID/TRAIN_ID
IF_TRAINED=COMPONENT_ROOT/'models/trained'/RUN_ID/PILOT_ID
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
Q4_ZIP=COMPONENT_ROOT/'data/raw/data_Q4_2021.zip' # Recorded for next notebook; do not check/open here.
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
complete_paths={
    'reservation_audit':AUDIT/'multimodel_coverage_complete.json',
    'if_pilot':PILOT/'pilot_complete.json',
    'matched_q3':COMPARE/'matched_comparison_complete.json',
    'transformer_training':TRAIN/'encoder_training_complete.json',
    'transformer_scoring':SCORE/'transformer_scoring_complete.json',
    'regime_audit':REGIME_AUDIT/'regime_audit_complete.json',
    'nuisance_ablation':ABLATION/'nuisance_regime_ablation_complete.json',
    'synthetic_sensitivity':SENS/'synthetic_sensitivity_complete.json',
    'streaming_replay':REPLAY/'streaming_replay_complete.json'}
records={key:read_json(path) for key,path in complete_paths.items()}
assert all(r['analysis_complete'] for r in records.values())
assert all(not r.get('future_test_opened',r.get('future_test_outcomes_opened',r.get('q4_outcomes_opened',False))) for r in records.values())
assert all(not r.get('sealed_June_test_opened',False) for r in records.values())
assert records['streaming_replay']['parity_drives']==50 and records['streaming_replay']['edge_case_checks_passed']
assert records['regime_audit']['regime_model_approved'] is False
assert records['nuisance_ablation']['regime_model_approved'] is False
reservation_path=COMPONENT_ROOT/records['reservation_audit']['split_path']
assert sha256(reservation_path)==records['reservation_audit']['split_sha256']
reservation=read_json(reservation_path)
assert reservation['future_test_outcomes_opened'] is False
models=list(reservation['models']);assert len(models)==5
future_test={str(s):m for m,g in reservation['models'].items() for s in g['future_test']}
development={str(s) for g in reservation['models'].values() for s in g['development']}
assert len(future_test)>100 and not set(future_test)&development
assert reservation['test_endpoint_start']=='2021-10-01' and reservation['test_endpoint_end']=='2021-12-31'
train_settings=read_json(TRAIN/'settings.json')
score_settings=read_json(SCORE/'settings.json')
if_settings=read_json(PILOT/'settings.json')
assert train_settings['models']==models and train_settings['window_days']==if_settings['window_days']==30
assert score_settings['detectors']==['temp_error','sector_error','joint_error']
assert score_settings['quantile']==if_settings['calibration_quantile']==.998
assert sha256(Q3_ZIP)==score_settings['q3_zip_sha256']
checkpoint=TRAINED/'best_model.pt';prep=TRAINED/'preprocessing.json'
if_model=IF_TRAINED/'if_models.joblib';if_spec=read_json(IF_TRAINED/'if_specification.json')
assert sha256(checkpoint)==records['transformer_training']['best_checkpoint_sha256']
assert sha256(prep)==records['transformer_training']['preprocessing_sha256']
assert sha256(if_model)==if_spec['artifact_sha256']
assert not if_spec['failure_labels_used'] and not if_spec['future_test_opened']
assert set(if_spec['thresholds'])==set(models)
trans_thresholds=read_json(SCORE/'calibrated_thresholds.json')
smart_thresholds=read_json(COMPARE/'baseline_calibration_thresholds.json')
assert set(trans_thresholds)==set(smart_thresholds)==set(models)
assert sha256(SCORE/'calibrated_thresholds.json')==records['transformer_scoring']['thresholds_sha256']
assert sha256(COMPARE/'matched_method_metrics.csv')==records['matched_q3']['metrics_sha256']
assert sha256(SENS/'controlled_sensitivity_summary.csv')==records['synthetic_sensitivity']['summary_sha256']
assert sha256(REPLAY/'telemetry_contract.json')==records['streaming_replay']['contract_sha256']
print('All prerequisite artifacts and reservation verified; Q4 was not read.')

All prerequisite artifacts and reservation verified; Q4 was not read.


## 1. Review full controlled sensitivity without tuning on failure labels

Notebook 23’s displayed table was truncated. Show a compact, complete pivot for the primary sector channel, temperature channel, joint channel, and fixed SMART sector change rule on the five injected perturbations. This is a diagnostic of sensor responsiveness, not an estimate of natural failure recall. The primary Q4 channel below remains the already declared sector reconstruction score regardless of this display.

In [2]:
sensitivity=pd.read_csv(SENS/'controlled_sensitivity_summary.csv')
assert len(sensitivity)==5*6*7
chosen_methods=['sector_error','temp_error','joint_error','sector_delta_fixed']
view=sensitivity.loc[sensitivity.method.isin(chosen_methods)&sensitivity.scenario.ne('unchanged')].copy()
view['new_alert_pct']=100*view.newly_flagged_fraction_of_initially_unflagged
pivot=view.pivot_table(index=['model','scenario'],columns='method',values='new_alert_pct',aggfunc='first')
display(pivot.round(1))
assert not view.new_alert_pct.isna().any()
print('Percentage of initially unflagged synthetic windows newly flagged. Q4 choice unchanged.')

method                                    joint_error  sector_delta_fixed  \
model                scenario                                               
HGST HMS5C4040BLE640 sector_onset_final         100.0               100.0   
                     sector_ramp_last7          100.0               100.0   
                     temp_flatline_last7          0.0                 0.0   
                     temp_shift_last7_6C          0.0                 0.0   
                     temp_spike_final_8C          4.0                 0.0   
ST12000NM0008        sector_onset_final         100.0               100.0   
                     sector_ramp_last7          100.0               100.0   
                     temp_flatline_last7          0.0                 0.0   
                     temp_shift_last7_6C          0.0                 0.0   
                     temp_spike_final_8C         19.1                 0.0   
ST4000DM000          sector_onset_final           0.5               100.0   
                     sector_ramp_last7            0.0               100.0   
                     temp_flatline_last7          0.0                 0.0   
                     temp_shift_last7_6C          0.0                 0.0   
                     temp_spike_final_8C          0.0                 0.0   
ST8000NM0055         sector_onset_final         100.0               100.0   
                     sector_ramp_last7          100.0               100.0   
                     temp_flatline_last7          0.0                 0.0   
                     temp_shift_last7_6C          0.0                 0.0   
                     temp_spike_final_8C          0.0                 0.0   
TOSHIBA MG07ACA14TA  sector_onset_final         100.0               100.0   
                     sector_ramp_last7          100.0               100.0   
                     temp_flatline_last7          0.0                 0.0   
                     temp_shift_last7_6C         10.7                 0.0   
                     temp_spike_final_8C         61.4                 0.0   

method                                    sector_error  temp_error  
model                scenario                                       
HGST HMS5C4040BLE640 sector_onset_final          100.0         0.0  
                     sector_ramp_last7           100.0         0.0  
                     temp_flatline_last7           0.0         0.0  
                     temp_shift_last7_6C           0.0         0.0  
                     temp_spike_final_8C           0.0         5.5  
ST12000NM0008        sector_onset_final          100.0         0.0  
                     sector_ramp_last7           100.0         0.0  
                     temp_flatline_last7           0.0         0.0  
                     temp_shift_last7_6C           0.0         0.5  
                     temp_spike_final_8C           0.0        45.7  
ST4000DM000          sector_onset_final            0.0         0.0  
                     sector_ramp_last7             0.0         0.0  
                     temp_flatline_last7           0.0         0.0  
                     temp_shift_last7_6C           0.0         8.5  
                     temp_spike_final_8C           0.0        88.4  
ST8000NM0055         sector_onset_final          100.0         0.0  
                     sector_ramp_last7           100.0         0.0  
                     temp_flatline_last7           0.0         0.5  
                     temp_shift_last7_6C           0.0         0.0  
                     temp_spike_final_8C           0.0        27.5  
TOSHIBA MG07ACA14TA  sector_onset_final          100.0         0.0  
                     sector_ramp_last7           100.0         0.0  
                     temp_flatline_last7           0.0         0.0  
                     temp_shift_last7_6C           0.0        17.8  
                     temp_spike_final_8C           0.0        64.0

Percentage of initially unflagged synthetic windows newly flagged. Q4 choice unchanged.


## 2. Freeze the Q4 outcome protocol and detector artifacts

Future-test drives are exactly the previously reserved July 1 serials. Features alone from Q3 September may warm the 30-day windows at the Q4 boundary, with the `failure` column never read during scoring. All Q4 model scores and flags must be hashed and locked **before any Q4 failure labels are read**. For endpoint targets, a positive has its first recorded failure within the next `h` days; a negative requires all following calendar days observed in Q4 with no failure; other endpoints are censored. An event is eligible for strict recall only when all `h` pre-failure daily score endpoints exist; early Q4 events lacking pre-October scores remain in the incomplete-event count. For 30-day targets, late December negatives are censored. No method may silently drop its missed or unscoreable failures.

The Q4 study is one-time, drive-disjoint and chronological. Q3 exploration influenced the selected primary channel and threshold policy; claim only Q4 future-test performance if Q4 has adequate events and coverage. Report all five models separately, including models with no complete events. No post-hoc per-model switching or Q4-informed threshold changes.

In [3]:
protocol={
    'id':LOCK_ID,'population':'exact July 1 reserved future_test serials, all five provisional HDD models',
    'reservation_sha256':sha256(reservation_path),'q3_archive_sha256':sha256(Q3_ZIP),
    'q4_archive_expected_relative_path':'data/raw/data_Q4_2021.zip',
    'q4_archive_contents_opened_by_this_notebook':False,
    'warmup':'Q3 September SMART features only for future_test drives; never Q3/Q4 failure flags during scoring',
    'scoring_endpoint_dates':['2021-10-01','2021-12-31'],
    'window_days':30,'require_consecutive_observed_valid_days':True,
    'primary_method':'transformer_sector_error',
    'primary_score':'mean squared normalized final-day hidden residual for SMART 5,197,198',
    'primary_alarm_rule':'score > frozen per-model higher 0.998 calibration quantile',
    'secondary_methods':['transformer_temp_error','transformer_joint_error'],
    'comparators':['isolation_forest','sector_positive_fixed','sector_delta_fixed',
                   'sector_positive_calibrated','sector_delta_calibrated'],
    'fixed_smart_rule':'score > 0','calibrated_smart_rule':'score > frozen per-model higher 0.998 quantile',
    'methods_on_identical_endpoints':True,
    'primary_horizon_days':14,'other_horizons_days':[7,30],
    'negative_rule':'all following h calendar days observed in Q4 and no first failure in horizon',
    'post_failure_rule':'first failure on/before endpoint is excluded from endpoint targets',
    'strict_event_rule':'all h pre-failure daily score endpoints present for every method',
    'incomplete_events':'preserve all recorded future-test failures and report incomplete/unscoreable separately',
    'metrics':['per-model and pooled descriptive average_precision when both classes present',
               'positive prevalence','positive endpoint recall','strict complete-history event recall',
               'false alerts per 1000 observed negative endpoints','all alerts per 1000 scored endpoints',
               'median first warning lead days','paired event outcomes'],
    '30_day_note':'do not compute AP/false-alert rate without observed negative follow-up',
    'uncertainty':'drive-level bootstrap intervals only if enough distinct positive drives; otherwise report undefined',
    'score_lock_before_labels':True,
    'q4_label_use':'one read after complete score manifest and protocol hash are frozen; no feedback to candidate selection',
    'winner_selection_after_q4':False,
    'regime_score_enabled':False,'drift_update_enabled':False,'automatic_healing_enabled':False,
    'ssd_generalization_claim':False,
    'limitations':['Q3 failures were inspected during development',
                   'four provisional HDD model names lack independently verified device type',
                   'the number of Q4 future-test failures is unknown']}
artifact_paths={
    'transformer_checkpoint':checkpoint,'transformer_preprocessing':prep,
    'transformer_training_settings':TRAIN/'settings.json',
    'transformer_scoring_settings':SCORE/'settings.json',
    'transformer_thresholds':SCORE/'calibrated_thresholds.json',
    'if_model':if_model,'if_specification':IF_TRAINED/'if_specification.json',
    'if_settings':PILOT/'settings.json',
    'smart_thresholds':COMPARE/'baseline_calibration_thresholds.json',
    'reservation':reservation_path,
    'stream_contract':REPLAY/'telemetry_contract.json'}
lock={'protocol':protocol,
      'artifacts':{key:{'path':str(path.relative_to(COMPONENT_ROOT)),'sha256':sha256(path)}
                   for key,path in artifact_paths.items()},
      'thresholds':{'transformer':trans_thresholds,'smart':smart_thresholds,
                    'isolation_forest':{m:float(if_spec['thresholds'][m]['threshold']) for m in models}},
      'future_test_drives_by_model':{m:len(reservation['models'][m]['future_test']) for m in models},
      'frozen_method_count':8,'failure_labels_used_for_thresholds':False,
      'q4_archive_opened':False,'q4_outcomes_opened':False}
lock_path=OUT/'q4_candidate_lock.json'
if lock_path.exists():assert read_json(lock_path)==lock,'Candidate lock changed; preserve it and investigate.'
else:lock_path.write_text(json.dumps(lock,indent=2))
protocol_path=OUT/'q4_blind_protocol.json'
if protocol_path.exists():assert read_json(protocol_path)==protocol
else:protocol_path.write_text(json.dumps(protocol,indent=2))
assert set(lock['artifacts'])==set(artifact_paths)
print('Locked primary method:',protocol['primary_method'])
print('Reserved drives by model:',lock['future_test_drives_by_model'])
print('Q4 archive and outcomes remain unopened.')

Locked primary method: transformer_sector_error
Reserved drives by model: {'TOSHIBA MG07ACA14TA': 6406, 'ST12000NM0008': 3776, 'ST4000DM000': 3758, 'ST8000NM0055': 2880, 'HGST HMS5C4040BLE640': 2531}
Q4 archive and outcomes remain unopened.


## 3. Closeout and next execution gate

Notebook 26 may read Q4 **feature columns only**, score the reserved drives using the locked artifacts, and write a complete score manifest. A later, separate evaluation pass may read the Q4 `failure` column only after confirming the score-lock hash. If Q4 has too few complete events, report that limitation; do not adapt the model, calibrations or target rules using Q4. The team's agent/frontend can consume Notebook 24’s separate score channels while a real schema mapping is agreed, but should not display the research flags as a calibrated failure probability.

In [4]:
assert sha256(lock_path)==sha256(OUT/'q4_candidate_lock.json')
assert sha256(reservation_path)==protocol['reservation_sha256']
for item in lock['artifacts'].values():
    path=COMPONENT_ROOT/item['path']
    assert sha256(path)==item['sha256']
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'candidate_lock_sha256':sha256(lock_path),'protocol_sha256':sha256(protocol_path),
        'reservation_sha256':sha256(reservation_path),
        'primary_method':protocol['primary_method'],'future_test_drives':len(future_test),
        'q4_archive_opened':False,'q4_outcomes_opened':False,'sealed_June_test_opened':False,
        'no_retraining_or_threshold_changes':True,'next_gate':'Q4 unlabeled score manifest, then Q4 labels'}
(OUT/'q4_protocol_lock_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 25 COMPLETE — Q4 primary candidate, comparators and evaluation protocol frozen.')
print('Q4 unopened. Lock:',lock_path)

NOTEBOOK 25 COMPLETE — Q4 primary candidate, comparators and evaluation protocol frozen.
Q4 unopened. Lock: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q4_blind_candidate_protocol_v1\q4_candidate_lock.json
